# 05. 점수표 — 실행 전·합치기 전 점검

대상: 팀원 4명 (11/25 회의에서 같이 본다. 18개를 돌리는 사람(팀장)은 돌리기 전에 꼭 실행)

11/24 팀장 결정(회의기록.md 'D6 취소'): 신호 18개 단독 점수표는 **팀장 한 명이 `--all` 로 한 번에** 돌린다 (계획서 §7 "사람별로 나누지 않는다" 그대로). 처음 적었던 4명 나눠 돌리기(D6)는 같은 날 취소했다.
이 노트북은 그 약속이 지켜지는지 **직접 돌려서** 확인한다. 결과 숫자(점수·Edge)는 여기서 보지 않는다 — 그것은 합친 뒤 `06_신호점수표.ipynb` 에서 본다.

1. 새 계산 부품 테스트 (대조군·t값·카이제곱·고르기 규칙·합치기 점검)
2. 내 컴퓨터 코드·데이터가 엔진 v2 고정값과 같은가 (11/24 '시가 0 매수' 결함을 고친 판. v1 기준 파일은 지우지 않고 보관) (다르면 실행이 거부된다)
3. (취소된) 4명 나눈 표 — 기록용
4. 결과 묶음 점검 (`results/scorecard_parts/`, 18개가 다 모였을 때 합치기 전에)
5. 시험 실행 기록 (팀장, 11/24)

이 노트북은 데이터 파일을 저장하지 않는다 (§9). 결과 파일은 명령(`python -m indexbeater optimize`)만 만든다.

In [1]:
# 이 칸을 맨 먼저 실행한다.
# 노트북이 어느 폴더에서 열려도, indexbeater 폴더가 있는 프로젝트 폴더를 찾아서 길 목록(sys.path)에 넣는다.
import os
import sys

root = os.getcwd()
while not os.path.isdir(os.path.join(root, "indexbeater")) and os.path.dirname(root) != root:
    root = os.path.dirname(root)          # 못 찾으면 한 칸 위 폴더로 올라가서 다시 찾는다
sys.path.insert(0, root)
os.chdir(root)
print("프로젝트 폴더:", root)

프로젝트 폴더: c:\Users\gram\Desktop\project\buysignal


## 1. 새 계산 부품 테스트

- `tests/test_metrics.py`: 대조군에서 신호 종목 빼기(중복_무시 포함), 대조 종목 없을 때 NaN, 상장폐지 최악(−100%), 5분위, 20일 블록 t값, 카이제곱, 승률·손익비, 상위 5종목, 체결 불가 비율, 최대 낙폭 — 손 계산과 비교
- `tests/test_optimize.py`: 이웃(한 축 한 칸), 중앙값으로 고르기, 300건 미만 제외, 동점 → 기본값 가까운 쪽, 전부 부족 → 기본값, 지문 비교, 나눈 표가 18개를 한 번씩 덮는지, 합치기 거부 조건

맨 아래 줄이 `passed` 이고 `failed` 가 없으면 통과.

In [2]:
import subprocess

r = subprocess.run([sys.executable, "-m", "pytest", "tests/test_metrics.py", "tests/test_optimize.py", "--color=no"],
                   capture_output=True, text=True, encoding="utf-8", errors="replace")
print(r.stdout[-1500:] or r.stderr[-1500:])
assert r.returncode == 0, "테스트 실패 — 팀장에게 알려 주세요"


................................                                         [100%]
32 passed in 1.74s



## 2. 내 컴퓨터가 엔진 v2 와 같은가

`results/engine_v2_reference.csv` 의 지문 (노트북 04 를 끝까지 돌리면 생긴다)(엔진 6개·데이터)과 지금 내 파일의 지문을 비교한다. 하나라도 다르면 `optimize` 명령이 실행을 거부한다.
새로 만든 `metrics.py`·`scorecard.py`·`scoring.py` 지문도 같이 보인다 — 결과 묶음에 적힌 지문과 같아야 한다 (합치기에서 확인).

In [3]:
import pandas as pd

from indexbeater import optimize as O
from indexbeater import scorecard as SC

now = SC.fingerprints(with_data=True)
ref = O.reference_fingerprints()
tbl = pd.DataFrame({"지금": now, f"엔진 {O.ENGINE_VERSION} 고정값": pd.Series(ref)})
tbl["같음"] = [(k not in ref) or (now.get(k) == ref.get(k)) for k in tbl.index]
display(tbl)
bad = O.check_frozen(now, ref)
print(f"엔진 {O.ENGINE_VERSION} 기준과 같음 ✓" if not bad else "다름 ✗ → " + "; ".join(bad))
print("git:", O.git_commit())

,지금,엔진 v2 고정값,같음
data/krx_adjusted.parquet,f73644072205,f73644072205,True
indexbeater/config.py,4fda0ea8b5a8,4fda0ea8b5a8,True
indexbeater/data_io.py,6c8f4c9b729e,6c8f4c9b729e,True
indexbeater/engine.py,c1feda245a03,c1feda245a03,True
indexbeater/exits.py,bbcbb26f4ab3,bbcbb26f4ab3,True
indexbeater/indicators.py,9a4a81a80982,9a4a81a80982,True
indexbeater/metrics.py,6de2262a060b,NaN,True
indexbeater/scorecard.py,2ecdc4cf9616,NaN,True
indexbeater/scoring.py,be21a465597f,NaN,True
indexbeater/signals.py,78db6954cfdf,78db6954cfdf,True


엔진 v2 기준과 같음 ✓
git: 2b54ee4 (고친 파일 있음)


## 2-2. 기간 설정 — 미래 자료를 못 보게 하는 장치 (팀장 점검, 2026-11-24)

점수표는 **설계 구간(2012-07-01 ~ 2019-12-31) 자료만** 쓴다. 시험 구간(2020년부터)은 열지 않는다 (기준 4장, 계획서 4.4).
막는 장치는 4겹이다 (④는 11/24 팀장 결정으로 더함).

| 장치 | 어디에 | 하는 일 |
|---|---|---|
| ① 시험 구간 잠금 | `data_io.load_long` | 2020-01-01 이후를 `allow_test=True` 없이 부르면 **파일을 읽기 전에** 멈춘다 |
| ② 설계 구간만 불러오기 | `optimize` → `load_wide()` | 기본값 `end=DESIGN_END`(2019-12-31) 그대로 부른다. 2020년 줄은 메모리에 올라오지도 않는다 |
| ③ 연도마다 자르기 (5.5) | `scorecard.Context.cut_idx` | 검증 연도 Y의 파라미터는 cut(= Y년 첫 거래일 − 21거래일)까지의 신호로만 고른다 |
| ④ 늦게 판 매매 빼기 (11/24 팀장 결정) | `scorecard._train_trades`, `metrics.ControlBase.matched_train` | 거래정지로 매도가 늦어져 **Y년 첫날 뒤에 판** 학습 매매를 신호 쪽·대조군 쪽 모두에서 뺀다 |

- 기간 값은 `config.py`에 있다. `config.py`는 지문으로 고정돼 있어서, 누가 기간을 바꾸면 2번 칸 지문이 달라지고 `optimize`가 실행을 거부한다. **그래서 기간은 따로 입력하지 않는다** (잘못 넣을 일이 없게).
- ③의 뜻: 신호가 cut날 → 다음 날 매수 → H 최대 20거래일 뒤 시가 매도 = **Y년 첫 거래일 시가**. 그래서 Y년 첫날 **다음** 가격은 Y년 세트를 고르는 데 쓰이지 않는다.

아래 첫 칸은 이것을 **가짜 주가**로 증명하는 테스트다 (`tests/check_period.py`, 10개).
가장 중요한 것: Y년 첫날 다음부터 가격을 마구 바꿔도 Y년까지의 학습 Edge·고른 세트가 한 칸도 안 바뀐다. 하루 앞(첫날 시가)부터 바꾸면 바뀐다 (= 이 점검이 진짜로 잡아낸다는 증거).
④는 종목 3개 중 1개를 Y년 첫날 앞뒤로 길게 거래정지시킨 가짜 주가로 따로 확인한다. ④를 빼고(고치기 전 방식) 돌리면 이 테스트가 **실패**한다 (팀장 확인, 11/24).
이 파일은 노트북 04의 pytest 개수(130)를 그대로 두려고 기본 pytest 모음에서 뺐다.


In [4]:
r = subprocess.run([sys.executable, "-m", "pytest", "tests/check_period.py", "-p", "no:cacheprovider",
                    "--color=no", "-W", "ignore"], cwd=root, capture_output=True, text=True,
                   encoding="utf-8", errors="replace")
print(r.stdout[-1500:])
print("종료 코드:", r.returncode, "(0이면 모두 통과)")


..........                                                               [100%]
10 passed in 29.57s

종료 코드: 0 (0이면 모두 통과)


실제 데이터에서 확인한다 (설계 구간만 읽는다, 1분쯤 걸린다).
- 불러온 마지막 날짜가 2020년 전인가
- 연도마다 cut 날짜와 '마지막 학습 매매를 파는 날'(= 검증 연도 첫 거래일)
- **④가 빼는 매매**: 거래정지가 길면 매도가 미뤄져 검증 연도 첫날 뒤에 판다. 유동성 통과 종목-일을 **전부 샀다고 치고** 그런 매매가 몇 건인지 센다 (신호 하나보다 훨씬 많은 경우를 센 것이다). 이 표의 '첫날 뒤에 판 매매'는 이제 학습에서 빠진다


In [5]:
import numpy as np

from indexbeater import config as C
from indexbeater import data_io
from indexbeater import engine as E
from indexbeater.exits import ExitRule

data = data_io.load_wide()                       # 기본값 = 2019-12-31 까지 (optimize 와 똑같이)
d = data["adj_close"].index
print("불러온 기간:", d[0].date(), "~", d[-1].date(), "| 2020년 전인가:", d[-1] < C.TEST_START)

m = E._Market(data)
allowed = data["tradable"].to_numpy(bool) & np.asarray(d >= C.SIGNAL_START)[:, None]
entry = pd.DataFrame(allowed, index=d, columns=m.codes)     # 유동성 통과 종목-일 전부 '신호'로
rows = []
for h in C.HOLDS:
    f = E.simulate_trades(entry, data, ExitRule(hold=h), dedupe=False, _m=m)
    f = f[f["status"] == E.FILLED]
    for y in SC.SELECT_YEARS:
        si = int(np.searchsorted(d, pd.Timestamp(f"{y}-01-01")))     # 검증 연도 첫 거래일 번호
        ci = si - C.WF_CUT_OFFSET                                      # cut = 첫 거래일 − 21거래일
        tr = f[f["t_idx"] <= ci]
        late = tr[tr["s_idx"] > si]
        rows.append({"검증연도": y, "H": h, "cut": d[ci].date(),
                     "마지막 학습매매 파는 날": d[si].date() if si < len(d) else "2020 첫날 (자료 밖 → 미완료로 빠짐)",
                     "학습 매매": len(tr), "첫날 뒤에 판 매매": len(late),
                     "비율 %": round(100 * len(late) / len(tr), 4),
                     "가장 늦게 판 날 (첫날+거래일)": int((late["s_idx"] - si).max()) if len(late) else 0,
                     "이유": late["reason"].value_counts().to_dict()})
period = pd.DataFrame(rows)
display(period)
print("첫날 뒤에 판 매매 비율 최대:", period["비율 %"].max(), "%")


불러온 기간: 2011-11-17 ~ 2019-12-30 | 2020년 전인가: True


,검증연도,H,cut,마지막 학습매매 파는 날,학습 매매,첫날 뒤에 판 매매,비율 %,가장 늦게 판 날 (첫날+거래일),이유
0,2015,5,2014-12-01,2015-01-02,385296,12,0.0031,232,{'S3': 12}
1,2016,5,2015-12-01,2016-01-04,599266,13,0.0022,167,{'S3': 13}
2,2017,5,2016-12-01,2017-01-02,830038,29,0.0035,573,{'S3': 29}
3,2018,5,2017-11-29,2018-01-02,1033154,47,0.0045,472,{'S3': 47}
4,2019,5,2018-11-29,2019-01-02,1270177,26,0.0020,229,{'S3': 26}
5,2020,5,2019-11-29,2020 첫날 (자료 밖 → 미완료로 빠짐),1491681,0,0.0000,0,{}
6,2015,10,2014-12-01,2015-01-02,385296,25,0.0065,232,{'S3': 25}
7,2016,10,2015-12-01,2016-01-04,599266,31,0.0052,167,{'S3': 31}
8,2017,10,2016-12-01,2017-01-02,830038,59,0.0071,573,{'S3': 59}
9,2018,10,2017-11-29,2018-01-02,1033144,95,0.0092,472,{'S3': 95}


첫날 뒤에 판 매매 비율 최대: 0.0213 %


**결과 읽기 (팀장, 2026-11-24 첫 실행)**

- 확인한 것: 시험 구간 잠금·설계 구간만 불러오기·연도마다 자르기·늦게 판 매매 빼기 4개 모두 동작한다. 가짜 주가 테스트 10개가 통과했다. 실제 데이터의 마지막 날은 2019-12-30이다.
- 2020용 세트(시험 구간에 쓸 값)는 2019-11-29까지의 신호로 고른다. 그 매매를 파는 날(2020년 첫날)은 불러오지 않았으므로 '미완료'로 빠진다 = 2020년 가격을 보지 않는다.
- 이상한 것: 거래정지가 길어서 검증 연도 첫날 **뒤에** 판 학습 매매가 있다 (종목-일 전부를 샀다고 쳐도 많아야 0.02%, 가장 늦은 것은 첫날 + 573거래일). 이 매매는 검증 연도 안의 가격을 쓴다. 계획서 코드(cut = days[days < start][-21], 신호일로 자르기)만으로는 막지 못한다.
- **팀장 결정 (11/24): ④로 고쳤다.** 학습 매매 = cut까지의 신호 중 **Y년 첫날 시가까지 판 매매**. 첫날 시가에 판 보통 매매(H=20)는 계획서대로 남는다. `scorecard.py`·`metrics.py` 지문이 바뀌었다 (고정 6개 파일·데이터 지문은 그대로). 11/25 4명 확인 필요.


## 3. (취소된) 4명 나눈 표 — 기록용

11/24 팀장 결정으로 **나눠 돌리지 않는다** (회의기록 'D6 취소'). 아래 표는 코드(`optimize.py` PARTS)에 남아 있어 보여 줄 뿐이다.
18개 돌리기: `python -m indexbeater optimize --all` → `python -m indexbeater optimize --merge` (프로젝트 폴더에서, 터미널로, 약 10분)

In [6]:
O.main(["--list"])

  1. 송재영: 15.vol_breakout, 18.candle, 17.obv, 13.squeeze, 6.sar  (세트 14개)
  2. 김동겸: 12.bollinger, 3.macd, 2.alignment, 14.donchian, 4.ichimoku  (세트 19개)
  3. 김승찬: 7.rsi, 16.volume_breakout, 5.adx_di, 8.stochastic  (세트 20개)
  4. 김정철: 9.disparity, 1.ma_cross, 10.cci, 11.williams_r  (세트 19개)


0

## 3-2. 기본값과 근거 (11/24 팀장 보완 — 4명 동의 필요)

기본값은 **동점일 때·모든 세트가 300건 미만일 때만** 쓴다 (5.5). 결정 ②(교과서 표준값)에서 근거가 약하던 7개를 논문·대표 자료로 다시 확인했다. 자료 값이 그리드 밖이면 그리드 안에서 가장 가까운 값을 쓴다 (그리드·config.py 는 그대로).

| 신호 | 전 → 후 | 근거 |
|---|---|---|
| ma_cross | 5·20 그대로 | 기재부 시사경제용어사전: 단기 골든크로스 = 5일선이 20일선 돌파 |
| adx_di | 25 그대로 | 와일더 규칙: 25 위 = 추세, 20 아래 = 추세 없음 |
| cci | 20 그대로 | 램버트(1980): 주기의 1/3, 보통 20 |
| disparity | 95% 그대로 | 키움증권 지표 안내: 20일 이격도 95% 밑돌면 매수 |
| volume_breakout | 60·3 → **60·2** | 오닐: 돌파 날 거래가 평균보다 40~50% 많게(1.4~1.5배) → 가장 가까운 2. N은 터틀 55일 → 60 |
| obv | 10 → **20** | 그랜빌 원래 OBV엔 평균선 없음 → 차트 프로그램 기본 20 (Barchart) |
| candle | 추세 5 → **10** | Marshall·Young·Rose(2006): 10일 지수평균으로 앞선 추세 판단 |

scorecard.py 가 바뀌어 지문이 달라졌다 → **돌리는 사람이 pull 한 뒤에 `--all` 을 돌린다** (`--merge` 는 scorecard.py 지문까지 같아야 합친다). 아래 칸: 18개 기본값이 모두 그리드 안에 있는지 확인.

In [7]:
# 기본값 18개가 그리드 안에 있는지, 그리드에서 몇 번째 세트인지 (그리드 밖이면 default_index 가 멈춘다)
rows = []
for name in SC.DEFAULT_PARAMS:
    sets = SC.param_sets(name)
    k = SC.default_index(name, sets)
    rows.append({"신호": name, "기본값": SC.DEFAULT_PARAMS[name], "세트 번호": k, "세트 수": len(sets)})
print("18개 모두 그리드 안:", len(rows) == 18)
print("scorecard.py 지문:", SC.fingerprints(with_data=False)["indexbeater/scorecard.py"])
pd.DataFrame(rows)

18개 모두 그리드 안: True
scorecard.py 지문: 2ecdc4cf9616


,신호,기본값,세트 번호,세트 수
0,ma_cross,"{'short': 5, 'long': 20}",0,8
1,alignment,"{'periods': (5, 20, 60)}",0,3
2,macd,"{'fast': 12, 'slow': 26, 'signal': 9}",3,4
3,ichimoku,"{'conv': 9, 'base': 26}",0,1
4,adx_di,"{'period': 14, 'adx_min': 25}",1,2
5,sar,"{'step': 0.02, 'max_step': 0.2}",0,1
6,rsi,"{'period': 14, 'low': 30}",6,8
7,stochastic,"{'k': 14, 'd': 3, 'low': 20, 'high': 80}",0,1
8,disparity,"{'ma': 20, 'thr': 0.95}",5,8
9,cci,{'period': 20},1,2


## 4. 결과 묶음 점검 (합치기 전에)

`--all` 이 끝나 `results/scorecard_parts/` 에 18개 묶음이 생긴 뒤 이 칸을 실행한다. 문제가 없으면 `python -m indexbeater optimize --merge`.
- 18개 신호가 다 있는가, 파일 7개가 다 있는가
- 모든 묶음의 지문(코드 9개·데이터)이 서로 같은가, 지금 내 파일과 같은가, 엔진 v2 와 같은가
- 포트폴리오를 건너뛴 묶음은 없는가

In [8]:
from pathlib import Path

part_dir = SC.PART_DIR
if not part_dir.exists():
    print("아직 결과 묶음이 없습니다:", part_dir)
else:
    problems, metas = O.check_parts(part_dir, now=now)
    rows = [{"신호": n, "돌린사람": m.get("돌린사람"), "git": m.get("git"), "돌린시각": m.get("돌린시각"),
             "걸린초": m.get("걸린초")} for n, m in metas.items()]
    if rows:
        display(pd.DataFrame(rows))
    print("합쳐도 됨 ✓" if not problems else "아직 합치면 안 됨 ✗\n - " + "\n - ".join(problems))

아직 결과 묶음이 없습니다: C:\Users\gram\Desktop\project\buysignal\results\scorecard_parts


## 4-2. 똑같이 나오는가 (재현 확인)

같은 엔진·같은 데이터·같은 프로그램 판이면 결과는 **소수점까지 똑같아야** 한다 (엔진에 무작위가 없고, 같은 값끼리 순서는 종목코드로 정한다 — 회의기록 11/24 재현 규칙).
다시 확인하고 싶을 때(필수 아님) `python -m indexbeater optimize --all --out results/scorecard_all` 로 한 번 더 돌린 뒤 이 칸을 실행해, 처음 돌린 묶음과 비교한다.
- 같음 = 통과 / 1e-9 이하 차이 = 프로그램 판 차이 (판을 기록) / 그보다 크거나 줄 수·빈칸 위치가 다름 = **코드 결함** → 결과를 쓰지 않는다

In [9]:
all_dir = Path("results/scorecard_all")
if part_dir.exists() and all_dir.exists():
    O.compare(part_dir, all_dir)
else:
    print("비교할 폴더가 아직 없습니다:", part_dir if not part_dir.exists() else all_dir)

비교할 폴더가 아직 없습니다: C:\Users\gram\Desktop\project\buysignal\results\scorecard_parts


## 5. 시험 실행 기록 (팀장, 2026-11-24)

명령이 끝까지 도는지, 파일이 다 생기는지만 확인했다. **결과 숫자는 보지 않았다** (결과를 보고 규칙을 바꾸지 않기 위해).

- 돌린 신호: ichimoku (S04) 1개, 결과는 임시 폴더(scratchpad/parts_smoke)에 저장 — `results/` 에는 아무것도 만들지 않음
- 걸린 시간: 약 103초 (대조군 바탕 약 60초 + 신호 약 32초 + 포트폴리오 6번)
- 파일 7개 모두 생김. 줄 수: 그리드 6 · 학습구간 36 · 고른 세트 36 · 점수표 12 · 검증 매매 55,396 · 포트폴리오 일별 7,362
- → 18개를 한 컴퓨터로 돌리면 약 20분, 4명이 나누면 1명당 약 5~7분 예상

**이상한 것 (실행 보류):** 자료에 `시가·고가·저가 = 0, 거래량 > 0` 인 칸이 53개(40종목) 있다. 엔진 v1 은 이날을 "살 수 있는 날"로 보고 **0원에 산다** → 수익이 무한대(inf)가 된다.
- 예: 003190 (2013-03-21) 등. 파는 쪽은 이미 막혀 있음(하한가처럼 처리).
- 그래서 점수표 코드에 **점검 장치**를 넣었다: 무한대 수익이 하나라도 있으면 숫자를 만들지 않고 멈춘다 (`BadTradeError`).
- 고치는 방법은 11/25 회의에서 4명이 정한다 (엔진 v2 로 바꾸면 `engine_v2_reference.csv` 새로 만들고 노트북 04 를 다시 돌린다).

**0원 매수가 신호마다 몇 건인가 (팀장, 11/24, 모든 세트 × H 5·10·20, 체결 매매만 셈):**

| 신호 | 0원 매수 | 신호 | 0원 매수 | 신호 | 0원 매수 |
|---|---|---|---|---|---|
| ma_cross | 3 | alignment | 0 | macd | 15 |
| ichimoku | 0 | adx_di | 0 | sar | 3 |
| rsi | 0 | stochastic | 3 | disparity | 9 |
| cci | 6 | williams_r | 0 | bollinger | 9 |
| squeeze | 0 | donchian | 0 | vol_breakout | 27 |
| volume_breakout | 0 | obv | 7 | candle | 0 |

- 합계 82건, 18개 중 9개 신호에 섞임 → 이대로는 9개 신호의 점수표가 멈춘다.
- 노트북 04 기준 매매(RSI 14·30, H=10)에는 0건.

**재검토 (팀장, 11/24 저녁) — 고친 것**
- 대조군 바탕(유동성 통과 전 종목)에도 0원 매수가 H마다 10건 있다 → 멈춤 장치 때문에 **지금은 18개 모두 3단계에서 멈춘다** (직접 돌려 확인: `--signals ichimoku` → "대조군 바탕 H=5: 무한대 10건", 종료코드 1, 결과 파일 안 만듦).
- `scorecard.py` 고른 세트의 학습거래수가 빈칸이면 멈추던 코드(`int(NaN or 0)`)를 고침.
- `optimize.py`: 신호 계산 중 멈춤도 깔끔히 알림, meta 에 프로그램 판(python·numpy·pandas·scipy·pyarrow) 기록, 합칠 때 판이 다르면 '주의' 표시, `--all`(18개 한 번에)·`--compare 폴더A 폴더B`(같은지 비교) 추가. 테스트 4개 추가 → 전체 126개 통과.

**재현 시험 (팀장, 11/24)** — 같은 엔진이면 똑같이 나오는가
- A: ichimoku 만 따로 / B: 다른 프로그램에서 cci 먼저 돌리고 ichimoku 를 두 번째로 (나눠 돌릴 때처럼 공통 준비를 이어 씀)
- 이 시험에서만 멈춤 장치를 껐다 (값은 쓰지 않고 같은지만 봄).
- 결과: 표 6개 모두 **"같음"** — 가장 큰 차이 0 (소수점 끝자리까지 같음). 걸린 시간 A 118초, B 174초.
- → 같은 컴퓨터에서는 돌리는 순서·묶음과 상관없이 똑같다. 다른 컴퓨터끼리는 4명이 돌린 뒤 `--compare` 로 확인한다.

**엔진 v2 로 고친 뒤 다시 확인 (팀장, 2026-11-24)** — 11/24 팀장 결정: 결함은 ① 엔진 v2, 기본값은 ② 교과서 표준값 (회의기록 '엔진 v2' 절)

- 고친 것: `engine.py` 만. 시가 0 인 날은 거래정지일처럼 사지도 팔지도 않는다 + 그날의 저가 0 으로 손절하지 않는다. 테스트 4개 추가 → 전체 130개 통과.
- 노트북 04: 2번 실행 모두 30/30 True. 새 기준 파일 `results/engine_v2_reference.csv` (engine.py `c1feda245a03`). v1 기준 파일은 그대로 보관.
- **18개 한 번에** (`--all`, 임시 폴더 scratchpad/all_v2): 끝까지 돎, 종료코드 0, 591초. 멈춤 장치(`BadTradeError`) 한 번도 안 걸림 → 0원 매수가 없어졌다.
  - 묶음 점검(`check_parts`): 18개 모두 있음, 파일 7개씩 다 있음, 지문 모두 같음 → 문제 없음.
- **나눠 돌리기 재현** (`--part 1` 을 따로, 임시 폴더 scratchpad/part1_v2, 246초): `--compare` 로 한 번에 돌린 것과 비교 → sar·squeeze·vol_breakout·obv·candle **5개 모두 같음** (vol_breakout 은 v1 에서 0원 매수가 가장 많던 신호).
- 이번에도 결과 숫자(점수·Edge)는 보지 않았다. `results/` 에는 v2 기준 파일 말고 아무것도 만들지 않았다.
- 남은 것: 4명이 각자 컴퓨터에서 노트북 04 → 이 노트북 → `--part 번호`. 다른 컴퓨터끼리 같은지는 합치기 전에 `--compare` 로 본다.
- **→ 바뀜 (11/24 팀장 결정, D6 취소):** 나눠 돌리지 않는다. 팀장이 `git pull` → pytest 130 → 이 노트북 → `--all` → `--merge`.
